In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0
from src.devices.qutip_device import HiddenDevice
from src.analysis.metrics import bell_fid, leakage
from src.analysis.plots import plot_unmodeled_sweep
from src.control.virtual_z import apply_frame
from src.control.waveforms import (
    build_local_drive_operators,
    generate_bell_state,
    build_bell_waveforms,
    pack_sequence,
)
from src.calibration.virtual_z_cal import calibrate_virtual_z

config = ExperimentConfig.load("configs/two_qubit_CZ_ideal.yaml")
dt = config.pulse.dt_ns
duration_ns = 20.0  # Single-qubit DRAG gate duration

# Load the Digital Twin's calibrated parameters & 40ns CZ pulse
cal_pulse_rad = np.load(config.artifact_path("cal_pulse_rad.npy"))
cal_g, cal_d, cal_a = np.load(config.artifact_path("cal_params.npy"))

H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=config.hardware.levels)
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=config.hardware.levels)
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)

# Calibrate Virtual-Z phases on the Digital Twin (frozen reference)
vz_cal = calibrate_virtual_z(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad,
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control
)
c1, c2 = vz_cal["v_phase1"], vz_cal["v_phase2"]

rho_twin = generate_bell_state(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad,
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control,
    v_phase1=c1, v_phase2=c2
)
fidelity_twin = bell_fid(rho_twin)

# Pack the fixed waveform sequence that we will replay on every device instance
wf_packed = pack_sequence(
    build_bell_waveforms(duration_ns, dt, cal_a, cal_d, cal_pulse_rad, v_phase2=c2)
)
rho00 = qt.ket2dm(qt.tensor(qt.basis(3, 0), qt.basis(3, 0)))
print(f"Digital Twin Reference Bell Fidelity: {fidelity_twin*100:.2f}%")

In [ ]:
zz_sweep_mhz = np.linspace(0.0, 2.5, 16)
zz_fidelities = []
zz_leakages = []

for zz in zz_sweep_mhz:
    # Explicitly set flux_bandwidth_mhz=None so we isolate ONLY parasitic ZZ crosstalk
    device = HiddenDevice.from_config(config, zz_crosstalk_mhz=float(zz), flux_bandwidth_mhz=None)
    rho_f = device.play_sequence(wf_packed, rho00)
    rho_framed = apply_frame(rho_f, c1, c2)
    
    zz_fidelities.append(bell_fid(rho_framed.full()))
    zz_leakages.append(leakage(rho_f.full()))

fig = plot_unmodeled_sweep(
    zz_sweep_mhz,
    zz_fidelities,
    zz_leakages,
    twin_fidelity=fidelity_twin,
    baseline_device_fidelity=zz_fidelities[0],
    xlabel=r"Unmodeled Static $ZZ$ Crosstalk $\zeta_{ZZ} / 2\pi$ (MHz)",
    title="Parasitic ZZ Crosstalk",
)
plt.show()

In [ ]:
# First visualize how the low-pass filter distorts the 40ns CZ pulse
t_cz = np.arange(len(cal_pulse_rad)) * dt
plt.figure(figsize=(7, 3.5))
plt.plot(t_cz, cal_pulse_rad / (2 * np.pi), "k--", linewidth=1, label="Commanded Pulse (Twin)")

freqs = [250.0, 100.0, 50.0]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(freqs)))

for bw in freqs:
    dev_tmp = HiddenDevice(flux_bandwidth_mhz=bw)
    filtered = dev_tmp.filter_flux_waveform(cal_pulse_rad)
    plt.plot(t_cz, filtered / (2 * np.pi), linewidth=1, label=f"Filtered ({bw:.0f} MHz BW)", color=colors[freqs.index(bw)])

plt.xlabel("Time (ns)")
plt.ylabel("Flux Amplitude (GHz)")
plt.title("Cryogenic Flux-Line Low-Pass Filtering on 40ns CZ Gate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Sweep bandwidth from 500 MHz (nearly ideal) down to 50 MHz (heavily distorted)
bw_sweep_mhz = np.linspace(500.0, 50.0, 60)
bw_fidelities = []
bw_leakages = []

for bw in bw_sweep_mhz:
    device = HiddenDevice(
        levels=config.hardware.levels,
        true_coupling=config.hardware.coupling_mhz,
        true_detuning=config.hardware.idle_detuning_mhz,
        true_anharm=config.hardware.anharmonicity_mhz,
        t1_ns=config.decoherence.t1_ns,
        t2_ns=config.decoherence.t2_ns,
        flux_bandwidth_mhz=float(bw),
    )
    rho_f = device.play_sequence(wf_packed, rho00)
    rho_framed = apply_frame(rho_f, c1, c2)
    
    bw_fidelities.append(bell_fid(rho_framed.full()))
    bw_leakages.append(leakage(rho_f.full()))

fig = plot_unmodeled_sweep(
    bw_sweep_mhz,
    bw_fidelities,
    bw_leakages,
    twin_fidelity=fidelity_twin,
    baseline_device_fidelity=zz_fidelities[0],
    xlabel="Flux-Line 3dB Bandwidth (MHz)",
    title="Flux Bandwidth Distortion",
    invert_xaxis=True,  # Left-to-right shows increasing distortion (decreasing bandwidth)
)
plt.show()